# Erratum: Motolinia (2021)

*Electoral Accountability and Particularistic Legislation: Evidence from an Electoral Reform in Mexico*

## Contents

- [1. Original claims and scope of the erratum](#section-1)
- [2. Replication data](#section-2)
- [3. Baseline replications](#section-3)
- [4. Erratum: Identification of the H2 final-year estimate](#section-4)
- [5. Supplementary analysis: Short-term reelection vs. term limits](#section-5)

**Findings.** The H1 and H3 point estimates are reproducible. In the H2 final-year sample, the difference-in-differences (DID) treatment variable is perfectly collinear with the state and month-by-year fixed effects. The supplementary comparison between the short-term reelection group and the term-limited group yields a negative point estimate.

**How to run.** Use the `.venv` interpreter in the parent `读论文` directory, where [`requirements.txt`](../requirements.txt) is located. Select that interpreter or the registered `reading-papers` kernel, then restart the kernel and run all cells. Section 2 downloads any missing files and verifies their SHA-256 hashes.

**Replication scope.** The Python implementation translates the author's Stata analysis and uses the published topic shares. The correlated topic model (CTM) is not refitted.

<a id="section-1"></a>

## 1. Original claims and scope of the erratum

The paper asks whether the opportunity for consecutive reelection leads legislators in a party-centered system to devote more attention to legislation that benefits specific places or groups. The outcome is the **share of discussion devoted to particularistic legislation in a legislative session**.

H1 compares the overall change in the long-term reelection group with that in the term-limited group. H2 asks whether this change is larger as elections approach. H3 compares the long-term and short-term reelection groups.

Page numbers refer to the printed journal version.

| Location | Comparison or claim | Notebook check |
| --- | --- | --- |
| p. 108, Table 2 | Long-term reelection vs. term limits (H1) | Replicate the baseline estimate |
| p. 109, Figure 5 | Pre-treatment cutoff checks for H1 | Reproduce the pre-treatment exercise |
| p. 110, Figure 6 | Estimates at different stages of the electoral cycle (H2) | Examine identification of the final-year estimate |
| p. 111, Table 3 | Long-term vs. short-term reelection (H3) | Replicate the baseline estimate |
| p. 110, Figure 7 | Pre-treatment cutoff checks for H3 | Reproduce the pre-treatment exercise |
| Supplementary comparison | Short-term reelection vs. term limits | Estimate the comparison directly |

<a id="section-2"></a>

## 2. Replication data

[Harvard Dataverse V1](https://doi.org/10.7910/DVN/NOMC0H).

The next cell verifies files that are already in the local `原作者文件/` data directory and downloads any missing files from Dataverse. If the project directory cannot be written to, the cell reports the fallback cache directory and uses it for the rest of the analysis. The files can also be downloaded directly from Dataverse and placed in the local data directory.

| Source file | Dataverse file ID | Purpose |
| --- | ---: | --- |
| `allres.csv` | 3892153 (`format=original`) | Session-level regression data, including shares for all 450 topics |
| `replication_analysis.do` | 3892146 | Regressions and figures in the article |
| `replication_appendix.do` | 3892135 | Appendix analyses |
| `replication_cleaning_CTM.R` | 3892151 | Text preprocessing, topic modeling, group definitions, and aggregation |
| `particularistic.csv` | 3892157 (`format=original`) | IDs of the 54 topics classified as particularistic |

In [ ]:
from pathlib import Path
from itertools import product
from functools import lru_cache
import re
import numpy as np
import pandas as pd
from scipy.linalg import qr
from scipy.stats import t as student_t
import statsmodels.api as sm
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from threadpoolctl import threadpool_limits

# Use one BLAS thread for these small matrix operations.
blas_limit = threadpool_limits(limits=1)

def find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        for candidate in [p, p / 'Motolinia(2021)']:
            if (candidate / 'Erratum-en.ipynb').is_file():
                return candidate
    raise FileNotFoundError('Open Erratum-en.ipynb from the Motolinia(2021) directory or the repository root.')

ROOT = find_root()
ORIGINAL = ROOT / '原作者文件'
pd.set_option('display.max_columns', 12)
pd.set_option('display.precision', 6)
plt.rcParams.update({'font.family': ['Times New Roman', 'DejaVu Serif'], 'mathtext.fontset': 'stix',
                     'figure.dpi': 110, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.titlesize': 15,
                     'axes.labelsize': 12, 'xtick.labelsize': 11, 'ytick.labelsize': 11})
COLORS = {'long': '#7CB6D3', 'short': '#8B1E2D', 'none': '#222222', 'muted': '#70869B'}

In [ ]:
import hashlib
import shutil
from tempfile import NamedTemporaryFile, gettempdir
from urllib.request import Request, urlopen

# Harvard Dataverse V1.0, DOI: 10.7910/DVN/NOMC0H
SOURCE_FILES = {
    'allres.csv': (3892153,
        '7b9d34c7fa32501435e5a3f3bc7b1ea9262fb321aa504c038aa652beded798aa'),
    'replication_analysis.do': (3892146,
        '074ebfd19d05c4d30c16f4118d820b929112be893c976288199bfe31f9725c54'),
    'replication_appendix.do': (3892135,
        'df377639185abde3bf226417b9b8ca7342859929fc2f0f700f8302682f6e1930'),
    'replication_cleaning_CTM.R': (3892151,
        '0170446dbce39b8985a37f616521637ab234ab514a5cc353b863d84c5ba10b6e'),
    'particularistic.csv': (3892157,
        '100b6b85ea03f16aa6a62dbbfa4d4c661cceef0f99bb115fb9bd1a228d6124f2'),
}


def verify_sha256(path, filename, expected):
    with path.open('rb') as source:
        actual = hashlib.file_digest(source, 'sha256').hexdigest()
    if actual != expected:
        raise ValueError(f'{filename} SHA-256 mismatch: expected {expected}, found {actual}')
    return actual


def prepare_data_directory(directory):
    # A complete local dataset only needs to be readable. Check write access if files are missing.
    if all((directory / name).is_file() for name in SOURCE_FILES):
        return directory
    directory.mkdir(parents=True, exist_ok=True)
    with NamedTemporaryFile(dir=directory, suffix='.part'):
        pass
    return directory


project_data = ROOT / '原作者文件'
try:
    ORIGINAL = prepare_data_directory(project_data)
except PermissionError:
    cache = Path(gettempdir()) / 'motolinia-NOMC0H-v1'
    try:
        ORIGINAL = prepare_data_directory(cache)
    except PermissionError as exc:
        raise PermissionError(f'Unable to write to the project data directory {project_data} or the fallback cache {cache}') from exc
    print(f'The project directory is not writable. Using the fallback data directory: {ORIGINAL}')
print('Filename\tSHA-256')
for filename, (file_id, expected) in SOURCE_FILES.items():
    target = ORIGINAL / filename
    if target.exists():
        actual = verify_sha256(target, filename, expected)
    else:
        url = f'https://dataverse.harvard.edu/api/access/datafile/{file_id}?format=original'
        request = Request(url, headers={'User-Agent': 'Mozilla/5.0 (reading papers)'})
        partial = None
        try:
            with NamedTemporaryFile(dir=ORIGINAL, suffix='.part', delete=False) as output:
                partial = Path(output.name)
                with urlopen(request, timeout=120) as response:
                    shutil.copyfileobj(response, output)
            actual = verify_sha256(partial, filename, expected)
            if target.exists():
                actual = verify_sha256(target, filename, expected)
            else:
                partial.replace(target)
        finally:
            if partial is not None:
                partial.unlink(missing_ok=True)
    print(f'{filename}\t{actual}')

### 2.1 Load the data

Each row represents a legislative session. `particularistic` is the sum of the shares of topics classified by the author as particularistic, expressed as a proportion from 0 to 1.

`treat4=1` denotes the long-term reelection group, and `treat2=1` denotes the short-term reelection group. Both variables equal zero for Mexico City (`CdMex`) and Puebla. Groups excluded from the relevant comparison are coded as missing.

`reform` follows the author's coding of legislative periods, so it can switch to 1 at different times across states. The Stata script constructs `date2` by multiplying `realyear` by 100 and appending the two-digit month, which yields `year * 10000 + month`. The Python code preserves that encoding for the original sample conditions and creates a conventional date for display and plotting.

In [ ]:
raw = pd.read_csv(ORIGINAL / 'allres.csv')
assert raw.shape == (6890, 471)
data = raw.copy()
date = pd.to_datetime(data.realyear.astype(str) + ' ' + data.month,
                      format='%Y %B', errors='coerce')
data['date'] = date
data['date2'] = date.dt.year * 10000 + date.dt.month
data['month_index'] = date.dt.year * 12 + date.dt.month
data['D_long'] = data.treat4 * data.reform
data['D_short'] = data.treat2 * data.reform
data['treat42'] = np.select([data.treat4.eq(1), data.treat2.eq(1)], [1., 0.], default=np.nan)
data['D_horizon'] = data.treat42 * data.reform
topic_columns = [f'V{i}' for i in range(1, 451)]
assert set(topic_columns).issubset(data.columns)
assert data.state.nunique() == 20
display(pd.DataFrame({'Variable': ['state', 'date', 'particularistic', 'treat4', 'treat2', 'reform'],
                      'Missing rows': [data[c].isna().sum() for c in
                                 ['state', 'date', 'particularistic', 'treat4', 'treat2', 'reform']]}))
display(data[['state', 'realyear', 'month', 'treat4', 'treat2', 'reform', 'particularistic']].head())

In [ ]:
def group_label(row):
    if row.treat4 == 1:
        return 'Long-term reelection'
    if row.treat2 == 1:
        return 'Short-term reelection'
    return 'Term-limited'

data['group'] = data.apply(group_label, axis=1)
group_inventory = data.groupby(['group', 'state']).agg(Sessions=('state', 'size'),
                  First_session=('date', 'min'),Last_session=('date', 'max')).reset_index()
display(group_inventory)
display(data.groupby('group').agg(States=('state', 'nunique'), Sessions=('state', 'size')))

### 2.2 From CTM shares to the outcome variable

The author's R script fits `stm(alldfm, K=450, seed=12345678)` and sums the shares of 54 topics using the classification file. Section 2 downloads and verifies the topic IDs in the published [`particularistic.csv`](https://dataverse.harvard.edu/api/access/datafile/3892157?format=original).

In [ ]:
particularistic_topics = pd.read_csv(ORIGINAL / 'particularistic.csv')['part'].tolist()
assert len(set(particularistic_topics)) == 54
rebuilt_y = raw[[f'V{i}' for i in particularistic_topics]].sum(axis=1)
max_y_gap = (rebuilt_y - raw.particularistic).abs().max()
topic_sum_gap = (raw[topic_columns].sum(axis=1) - 1).abs().max()
# Allow for rounding in the CSV export. Tolerances are in proportion units.
assert np.allclose(rebuilt_y, raw.particularistic, atol=1e-6, rtol=0)
assert np.allclose(raw[topic_columns].sum(axis=1), 1, atol=1e-5, rtol=0)
display(pd.DataFrame({'Check': ['Maximum deviation of the 450-topic sum from 1',
                              'Maximum difference between the 54-topic sum and the published outcome'],
                      'Value': [topic_sum_gap, max_y_gap]}))

The regressions use the author's `particularistic` column as published. The largest difference between that column and the reconstructed sum is about $4.94\times10^{-7}$ in proportion units. Across all 450 topics, the largest deviation of the sum from 1 is about $1.615\times10^{-6}$. These discrepancies are consistent with rounding during data export.

<a id="section-3"></a>

## 3. Baseline replications

### 3.1 The original model and its Python implementation

The paper's model is

$$Y_{ist}=\alpha+\beta D_{st}+\mu_s+\gamma_t+\varepsilon_{ist}.$$

$\mu_s$ and $\gamma_t$ are state and month-by-year fixed effects. The treatment variable $D$ is the product of the relevant group indicator $R_s$ and the author's `reform` indicator $T_t$.

The next cell prints the author's regression commands and compares them with the Python implementation.

In [ ]:
author_code = (ORIGINAL / 'replication_analysis.do').read_text(encoding='utf-8')
for marker in ['*Table 2: H1', '*Table 3: H3']:
    block = author_code[author_code.index(marker):].splitlines()
    command = next(line.strip() for line in block if line.strip().startswith('reg '))
    print(marker, '\n', command, '\n')
scope_comparison = pd.DataFrame([
    {'Analysis':'H1','Article location':'Table 2, p.108','Original treatment variable':'timb3=treat4*reform',
     'Python treatment variable':'D_long','Sample rule':'Groups covered by treat4, excluding missing values as in the original'},
    {'Analysis':'H3','Article location':'Table 3, p.111','Original treatment variable':'c.treat42#c.reform',
     'Python treatment variable':'D_horizon','Sample rule':'Long-term and short-term reelection groups'},
    {'Analysis':'H2 final year','Article location':'Figure 6, p.110','Original treatment variable':'timb3, if timingc==1',
     'Python treatment variable':'D_long with the original timingc filter','Sample rule':'Parse the six replace conditions in the .do file'},
    {'Analysis':'Supplementary comparison','Article location':'Added here','Original treatment variable':'Not reported in the article',
     'Python treatment variable':'D_short','Sample rule':'Reestimate using the short-term and term-limited groups'}
])
display(scope_comparison)
display(pd.DataFrame([
    {'Setting':'Fixed effects','Original':'State + month-by-year','Python':'Same fixed-effects space'},
    {'Setting':'Standard errors','Original':'Clustered by state','Python':'State-clustered CR1, t(G−1) reference distribution'},
    {'Setting':'Data and outcome','Original':'allres.csv / particularistic','Python':'Original values and treatment coding retained'},
    {'Setting':'Perfect collinearity','Original':'Stata can omit redundant regressors','Python':'Check residual variation and rank. Report effects when identified.'},
    {'Setting':'Supplementary bootstrap','Original':'Python implementation','Python':'Used for the short-term reelection comparison'}
]))

### 3.2 Python estimation

The implementation uses Frisch–Waugh–Lovell residualization. QR decomposition retains a basis for the fixed effects. If treatment is perfectly collinear with those effects, the routine marks the coefficient as unidentified.

Standard errors use the state-clustered CR1 covariance estimator. Conventional p-values and 95% intervals use the $t(G-1)$ reference distribution. For the supplementary comparison with few clusters, the null-imposed Rademacher wild cluster bootstrap enumerates all $2^G$ sign combinations.

In [ ]:
@lru_cache(None)
def sign_weights(g):
    if g > 12:
        raise ValueError('The enumerated bootstrap is limited to the supplementary comparison with few clusters.')
    return np.asarray(list(product([-1., 1.], repeat=g))).T

class FixedEffectsDesign:
    def __init__(self, frame):
        self.frame = frame.dropna(subset=['state', 'date2', 'particularistic']).copy()
        self.n = len(self.frame)
        self.cluster_id, self.states = pd.factorize(self.frame.state, sort=True)
        self.g = len(self.states)
        self.cluster = np.eye(self.g)[self.cluster_id]
        months, _ = pd.factorize(self.frame.date2, sort=True)
        # Use all state dummies to span the intercept and omit one month-by-year category.
        self.controls = np.column_stack([self.cluster,
                            np.eye(months.max()+1)[months, 1:]])
        q, r, piv = qr(self.controls, mode='economic', pivoting=True)
        diagonal = np.abs(np.diag(r))
        tol = max(self.controls.shape) * np.finfo(float).eps * diagonal.max()
        self.rank = int((diagonal > tol).sum())
        self.q = q[:, :self.rank]
        self.y = self.frame.particularistic.to_numpy(float)
        self.yr = self.residualize(self.y)

    def residualize(self, z):
        z = np.asarray(z, dtype=float)
        return z - self.q @ (self.q.T @ z)

    def fit(self, regressor, wild=False, return_boot=False):
        if isinstance(regressor, str):
            regressor = self.frame[regressor]
        elif isinstance(regressor, pd.Series):
            regressor = regressor.reindex(self.frame.index)
        x = np.asarray(regressor, dtype=float)
        assert len(x) == self.n and np.isfinite(x).all()
        xr = self.residualize(x)
        xx = float(xr @ xr)
        result = dict(n=self.n, states=self.g, rank_FE=self.rank,
                      rank_FE_D=self.rank + int(xx > 1e-10), residual_D_ss=xx)
        if xx <= 1e-10:
            result.update(identified=False, b=np.nan, se=np.nan, p=np.nan,
                          ci_lower=np.nan, ci_upper=np.nan, wild_p=np.nan)
            return result
        b = float(xr @ self.yr / xx)
        u = self.yr - b*xr
        correction = self.g/(self.g-1) * (self.n-1)/(self.n-self.rank-1)
        cluster_scores = self.cluster.T @ (xr*u)
        se = float(np.sqrt(correction * (cluster_scores @ cluster_scores)) / xx)
        p = float(2*student_t.sf(abs(b/se), self.g-1))
        critical = student_t.ppf(.975, self.g-1)
        result.update(identified=True, b=b, se=se, p=p,
                      ci_lower=b-critical*se, ci_upper=b+critical*se, wild_p=np.nan)
        if wild:
            # H0: beta = 0. The residuals from the restricted fixed-effects fit are yr.
            weights = sign_weights(self.g)
            y_star = self.yr[:, None] * weights[self.cluster_id]
            y_star = self.residualize(y_star)
            b_star = (xr @ y_star) / xx
            u_star = y_star - xr[:, None]*b_star
            scores = self.cluster.T @ (xr[:, None]*u_star)
            se_star = np.sqrt(correction*np.sum(scores**2, axis=0)) / xx
            boot_t = np.abs(b_star/se_star)
            result['wild_p'] = float(np.mean(boot_t >= abs(b/se)-1e-12))
            if return_boot:
                result['boot_t'] = boot_t
        return result

def design_for(column, frame=data):
    return FixedEffectsDesign(frame.dropna(subset=[column]))

def result_table(records):
    table = pd.DataFrame(records).copy()
    for source, target in [('b','Estimate (pp)'), ('se','Clustered SE (pp)'),
                            ('ci_lower','95% lower bound'), ('ci_upper','95% upper bound')]:
        table[target] = table[source]*100
    return table[['comparison','n','states','identified','Estimate (pp)','Clustered SE (pp)',
                  '95% lower bound','95% upper bound','p','wild_p','rank_FE','rank_FE_D']].rename(
                  columns={'comparison':'Comparison','n':'Sessions','states':'States','identified':'Identified',
                           'p':'Conventional p','wild_p':'wild bootstrap p'})

def forest(ax, records, labels, title, colors):
    for i, (r, label, color) in enumerate(zip(records, labels, colors)):
        if r['identified']:
            ax.errorbar(r['b']*100, i,
                xerr=[[100*(r['b']-r['ci_lower'])], [100*(r['ci_upper']-r['b'])]],
                fmt='o', color=color, capsize=4, markersize=7)
    ax.axvline(0, color='#555555', linestyle='--', linewidth=1)
    ax.set_yticks(range(len(labels)), labels)
    ax.set_ylim(len(labels)-.5, -.5)
    ax.set_xlabel('DID coefficient (percentage points); 95% clustered CI')
    ax.set_title(title, loc='left', pad=14)
    ax.grid(axis='x', color='#E4EAF0', linewidth=.7)

### 3.3 Baseline replications for H1 and H3

This section checks the main point estimates and the independent variation in the treatment variables before examining the H2 final-year estimate.

In [ ]:
h1_design = design_for('D_long')
h3_design = design_for('D_horizon')
h1 = h1_design.fit('D_long'); h1['comparison'] = 'H1: Long-term reelection vs. term limits'
h3 = h3_design.fit('D_horizon'); h3['comparison'] = 'H3: Long-term vs. short-term reelection'
assert h1['n'] == 4899 and np.isclose(h1['b'], .01525336444865884)
assert h3['n'] == 6062 and np.isclose(h3['b'], .02623265672428348)
display(result_table([h1, h3]))

# Cross-check FWL against full OLS with clustered covariance in statsmodels.
X_check = np.column_stack([h1_design.controls, h1_design.frame.D_long])
ols_check = sm.OLS(h1_design.y, X_check).fit(cov_type='cluster',
    cov_kwds={'groups': h1_design.frame.state}, use_t=True)
assert np.isclose(ols_check.params[-1], h1['b'], atol=1e-12)
# Compute the contrast for the treatment coefficient only.
h1_contrast = ols_check.t_test(f'{ols_check.model.exog_names[-1]} = 0')
assert np.isclose(h1_contrast.sd.item(), h1['se'], atol=1e-12)
fig, ax = plt.subplots(figsize=(10, 3.3), layout='constrained')
forest(ax, [h1,h3], ['H1: Long-term reelection vs. term limits','H3: Long-term vs. short-term reelection'],
       'Baseline replication', [COLORS['long'], COLORS['short']])
plt.show()

The H1 estimate is approximately **+1.53 percentage points**. Under the CR1 estimator and $t(G-1)$ reference distribution used here, its p-value is approximately **0.050008**, just above 0.05. The H3 estimate is approximately **+2.62 percentage points**. Both point estimates and standard errors agree with the paper at its reporting precision, and both treatment variables have variation beyond the fixed effects.

**Comparison with the published results (percentage points)**

| Result | Published | Replicated | Comparison |
| --- | ---: | ---: | --- |
| H1 (Table 2, p. 108) | +1.53 | +1.5253 | Matches at the reported precision |
| H3 (Table 3, p. 111) | +2.62 | +2.6233 | Matches at the reported precision |
| H2 final year (Figure 6, p. 110) | About +4.60 (read from the figure) | +4.5992 (retain $D$ and additionally omit the June 2016 dummy) | Matches the plotted value under that constraint. $\beta$ is unidentified in the full specification. |

The published H1 and H3 values are 0.0153 and 0.0262 in proportion units. The H2 comparison concerns the numerical value of the plotted final-year point. See Section 4.6 for the parameterizations.

### 3.4 Pre-treatment checks for H1 and H3

The code extracts state election years from the `rm101` date conditions in the original `.do` file and constructs the 12 + 12 + 9 cutoffs in the same order. Each estimate is shown with its conventional 95% confidence interval.

In [ ]:
author_code = (ORIGINAL / 'replication_analysis.do').read_text(encoding='utf-8')
cohort = {}
for threshold, expression in re.findall(
        r'(?:gen|replace) rm10[^=]*=0 if date2<(\d+)[^&]*& \(([^\r\n]*)\)', author_code):
    for state in re.findall(r'state=="([^"]+)"', expression):
        cohort[state] = int(threshold[:4])
assert len(cohort) == 20 and cohort['Puebla'] == 2013 and cohort['CdMex'] == 2015
cutoffs = [(shift, month) for shift, count in [(2,12),(1,12),(0,9)]
                         for month in range(1,count+1)]

def placebo_checks(group_column, wild=False):
    sample = data.loc[data.reform.eq(0) & data[group_column].notna()].copy()
    design = FixedEffectsDesign(sample)
    records, draws = [], []
    for i, (shift, month) in enumerate(cutoffs, 1):
        threshold = (design.frame.state.map(cohort)-shift)*10000+month
        x = design.frame[group_column] * design.frame.date2.ge(threshold)
        r = design.fit(x, wild=wild, return_boot=wild)
        if wild:
            draws.append(r.pop('boot_t'))
        r.update(cutoff=i, cohort_shift=shift, month=month)
        records.append(r)
    table = pd.DataFrame(records)
    joint_p = np.nan
    if wild:
        # Use the same enumerated weights and state ordering for every cutoff to retain dependence across tests.
        observed = np.max(np.abs(table.b/table.se))
        joint_p = float(np.mean(np.max(np.vstack(draws),axis=0) >= observed-1e-12))
    return table, joint_p

pre_h1, _ = placebo_checks('treat4')
pre_h3, _ = placebo_checks('treat42')
display(pd.DataFrame({'Comparison':['H1: Long-term reelection vs. term limits','H3: Long-term vs. short-term reelection'],
    'Cutoffs':[len(pre_h1),len(pre_h3)],
    'Cutoffs with conventional p < 0.05':[(pre_h1.p<.05).sum(),(pre_h3.p<.05).sum()]}))

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True, layout='constrained')

for ax, table, did, title, color in zip(
        axes, [pre_h1, pre_h3], [h1, h3],
        ['H1: Long-term reelection vs. term limits', 'H3: Long-term vs. short-term reelection'],
        [COLORS['long'], COLORS['short']]):

    ax.errorbar(
        table.cutoff, table.b * 100,
        yerr=np.vstack([
            (table.b - table.ci_lower) * 100,
            (table.ci_upper - table.b) * 100,
        ]),
        fmt='s', ms=4, capsize=2, color=COLORS['muted'],
    )

    did_position = len(table) + 1
    ax.axvline(did_position - 0.5, color='#B2414C', linewidth=1)
    ax.errorbar(
        did_position, did['b'] * 100,
        yerr=[
            [100 * (did['b'] - did['ci_lower'])],
            [100 * (did['ci_upper'] - did['b'])],
        ],
        fmt='s', ms=6, capsize=3, color=color,
    )

    ax.axhline(0, color='#777777', linestyle='--', linewidth=1)
    ax.set_title(title, loc='left')
    ax.set_ylabel('Estimate (pp)')

axes[-1].set_xlabel('Pre-treatment cutoff and main DID estimate')
axes[-1].set_xticks(
    [1, 6, 12, 18, 24, 30, 33, 34],
    ['1', '6', '12', '18', '24', '30', '33', 'DID'],
)
plt.show()

The plots reproduce the pre-treatment exercise and the main DID points shown in Figures 5 and 7. At the 5% level, the replication flags 4 of 33 H1 cutoffs, while the article reports 2. For H3, both report 0 of 33.

<a id="section-4"></a>

## 4. Erratum: Identification of the H2 final-year estimate

H2 predicts a larger effect as the election approaches. Figure 6 does not show a monotonic increase: the $t+1$ point estimate is below the estimate at $t$. The claim that the effect is largest in the final year therefore depends on $t+2$. Sections 4.4–4.6 show that the treatment variable for this sample is perfectly collinear with the state and month-by-year fixed effects. This undermines the evidence for H2 in Figure 6.

### 4.1 Reconstruct the three electoral-cycle samples

The `timinga`, `timingb`, and `timingc` conditions come directly from the author's `.do` file and correspond to `Election (t)`, `t+1`, and `t+2` in Figure 6.

In [ ]:
start = author_code.index('*Figure 6: H2')
end = author_code.index('coefplot', start)
print(author_code[start:end].strip())

def original_timing_mask(name):
    clauses = re.findall(r'replace '+name+r'=1 if date2>(\d+) & date2<=(\d+) & \(([^\r\n]*)\)',
                         author_code)
    assert len(clauses) == 6
    mask = pd.Series(False, index=data.index)
    for lo, hi, expression in clauses:
        states = re.findall(r'state=="([^"]+)"', expression)
        mask |= data.date2.gt(int(lo)) & data.date2.le(int(hi)) & data.state.isin(states)
    return mask

timing = {name: original_timing_mask(name) for name in ['timinga','timingb','timingc']}
h2_records, h2_designs = [], {}
for name, label in [('timinga','Election (t)'),('timingb','t+1'),('timingc','t+2')]:
    model = FixedEffectsDesign(data.loc[timing[name] & data.D_long.notna()])
    h2_designs[name] = model
    r = model.fit('D_long');r['comparison'] = label
    h2_records.append(r)
display(result_table(h2_records))
assert h2_records[2]['n'] == 1511 and not h2_records[2]['identified']

The $t+2$ coefficient is blank because the treatment variable has no variation outside the full fixed-effects space. The next sections show which observations are available and how an additional omission constraint reproduces the plotted value of about +4.60 percentage points.

### 4.2 Gaps in sample coverage

In [ ]:
h2 = h2_designs['timingc'].frame.copy()
h2['D'] = h2.D_long
h2['L'] = h2.date2.between(20170007,20180006).astype(int)
h2['E'] = h2.date2.between(20120007,20130006).astype(int)
h2['P'] = h2.state.eq('Puebla').astype(int)
windows = [(20120007,20130006),(20140007,20150006),
           (20150007,20160006),(20170007,20180006)]
window_labels = ['Jul 2012–Jun 2013','Jul 2014–Jun 2015',
                 'Jul 2015–Jun 2016','Jul 2017–Jun 2018']
group_masks = [h2.treat4.eq(1),h2.state.eq('CdMex'),h2.state.eq('Puebla')]
coverage = pd.DataFrame([[int((mask & h2.date2.between(lo,hi)).sum())
                          for lo,hi in windows] for mask in group_masks],
                          index=['Long-term reelection','Mexico City','Puebla'],columns=window_labels)
assert coverage.to_numpy().tolist() == [[0,491,346,492],[0,81,0,0],[69,0,0,32]]
display(coverage)
fig, ax = plt.subplots(figsize=(11,3.5),layout='constrained')
ax.imshow(coverage, cmap='Blues', vmin=0, vmax=coverage.to_numpy().max(),aspect='auto')
for i in range(3):
    for j in range(4):
        n = int(coverage.iloc[i,j])
        ax.text(j,i,str(n) if n else '—',ha='center',va='center',fontsize=17,
                color='white' if n>300 else '#172A3D')
ax.set_xticks(range(4),[s.replace('–','–\n') for s in window_labels])
ax.set_yticks(range(3),coverage.index)
ax.set_title('H2 final-year sample: Sessions by selected calendar window (— = no observations)',loc='left')
plt.show()

cdmex_coding = h2.loc[h2.state.eq('CdMex')].groupby('reform').size().rename('Sessions')
display(cdmex_coding)

Puebla supplies the only observations from the term-limited group in the last window. Its earlier comparison observations are from 2012/13, with none in the middle window. Mexico City contributes observations in 2014/15 but none in 2017/18.

### 4.3 Compare with a standard DID design

The calculation below takes the **double difference of the treatment variable $D$** to check its independent variation. In the standard two-group, two-period example, the treatment matrix is $\begin{pmatrix}0&1\\0&0\end{pmatrix}$, which gives $(1-0)-(0-0)=1$.

Suppose $D_{st}=a_s+b_t$, where $a_s$ is constant over time within a state and $b_t$ is common to all states in a given month and year. The change in each group would then be $b_1-b_0$, making the double difference zero. The standard DID example gives 1, so that additive representation is impossible. In the original H2 final-year sample, however, one choice of $a_s$ and $b_t$ fits every observed row: $a_s=0$ for the long-term reelection group and Mexico City, $a_s=-1$ for Puebla, $b_t=1$ in the earliest and latest windows, and $b_t=0$ in the middle window.

In [ ]:
S = np.array([1.,1.,0.,0.]); T = np.array([0.,1.,0.,1.]); D = S*T
F_standard = np.column_stack([np.ones(4),S,T])
standard_residual = D-F_standard@np.linalg.lstsq(F_standard,D,rcond=None)[0]
assert np.isclose(standard_residual@standard_residual,.25)
display(pd.DataFrame([
 {'Comparison':'Standard 2×2 example','Rank of FE':np.linalg.matrix_rank(F_standard),
  'Rank of FE + D':np.linalg.matrix_rank(np.column_stack([F_standard,D])),
  'Residualized D sum of squares':standard_residual@standard_residual},
 {'Comparison':'Original H2 final-year sample','Rank of FE':59,'Rank of FE + D':59,
  'Residualized D sum of squares':h2_records[2]['residual_D_ss']}
]))

In [ ]:
# Build the right panel from the actual H2 sample and leave unobserved cells as NaN.
from matplotlib.colors import ListedColormap, BoundaryNorm
display_frame = h2.assign(
    display_group=np.select([h2.treat4.eq(1),h2.state.eq('CdMex')],
                            ['Long-term reelection','Mexico City'],default='Puebla'),
    calendar=np.select([h2.E.eq(1),h2.L.eq(1)],['2012/13','2017/18'],default='2014–16'))
assert display_frame.groupby(['display_group','calendar']).D.nunique().le(1).all()
observed_matrix = display_frame.pivot_table(index='display_group',columns='calendar',
                      values='D',aggfunc='first').reindex(
                      index=['Long-term reelection','Mexico City','Puebla'],
                      columns=['2012/13','2014–16','2017/18'])
assert np.array_equal(h2.D, -h2.P + (h2.L+h2.E))
cmap = ListedColormap(['#EDF1F4','#7CB6D3']); cmap.set_bad('#FFFFFF')
fig, axes = plt.subplots(1,2,figsize=(12,4.8),layout='constrained')
panels = [(np.array([[0.,1.],[0.,0.]]),['Treatment','Control'],['Before','After'],
           'Standard DID: Independent treatment variation'),
          (observed_matrix.to_numpy(),['Long-term reelection (a=0)','Mexico City (a=0)','Puebla (a=−1)'],
           ['2012/13\nb=1','2014–16\nb=0','2017/18\nb=1'],
           'H2 final-year sample: D is additive')]
for ax, (values, row_names, column_names, title) in zip(axes,panels):
    ax.imshow(np.ma.masked_invalid(values),cmap=cmap,
              norm=BoundaryNorm([-.5,.5,1.5],2),aspect='auto')
    for i in range(values.shape[0]):
        for j in range(values.shape[1]):
            val=values[i,j]
            ax.text(j,i,'—' if np.isnan(val) else str(int(val)),ha='center',va='center',fontsize=22)
    ax.set_xticks(range(len(column_names)),column_names)
    ax.set_yticks(range(len(row_names)),row_names)
    ax.set_title(title,loc='left',fontsize=13,pad=12)
axes[0].set_xlabel(r'$(1-0)-(0-0)=1\ne0$',fontsize=18,labelpad=13)
axes[1].set_xlabel(r'$D_{st}=a_s+b_t,\quad\widetilde D=0$',fontsize=18,labelpad=13)
plt.show()

### 4.4 Row-by-row identity and matrix rank

Let $L_t$ indicate July 2017 to June 2018, $E_t$ indicate July 2012 to June 2013, and $P_s$ indicate Puebla. In this subsample,

$$D_{st}=R_sT_{st}=L_t+E_t-P_s.$$

$L_t$ and $E_t$ can be expressed using month-by-year dummies, and $P_s$ is a state dummy. Thus $D$ lies in the span of the fixed effects. The next cell checks this identity for every row and compares the matrix rank before and after adding $D$.

In [ ]:
identity_error = h2.D - (h2.L+h2.E-h2.P)
assert (identity_error == 0).all()
identity_rows = h2.groupby(['group','reform','L','E','P'],dropna=False).agg(
    N=('D','size'),D=('D','first')).reset_index()
identity_rows['L+E-P'] = identity_rows.L+identity_rows.E-identity_rows.P
display(identity_rows)
display(pd.DataFrame([
    {'Comparison':'Full H1 sample','N':h1['n'],'Rank of FE':h1['rank_FE'],'Rank of FE + D':h1['rank_FE_D'],
     'Residualized D sum of squares':h1['residual_D_ss']},
    {'Comparison':'H2 final-year sample','N':len(h2),'Rank of FE':h2_records[2]['rank_FE'],
     'Rank of FE + D':h2_records[2]['rank_FE_D'],'Residualized D sum of squares':h2_records[2]['residual_D_ss']}
]))

### 4.5 Perfect collinearity

Campeche and July 2012 are the reference state and month. After the usual reference categories are omitted, the fixed-effects matrix has full column rank. Retaining $D$ and omitting an additional month in the middle window imposes another constraint.

When the column selection is reconstructed in the order of the public script, retaining the intercept, $D$, state dummies, and month-by-year dummies in ascending order makes June 2016 redundant. Omitting that column reproduces the plotted +4.60 percentage points. In the H2 final-year sample, the DID treatment variable is perfectly collinear with the state and month-by-year fixed effects.

In [ ]:
def explicit_fe(frame, state_base=None, month_base=None):
    states = pd.get_dummies(frame.state, prefix='s', dtype=float)
    months = pd.get_dummies(frame.date2.astype(int), prefix='m', dtype=float)
    state_ref = states.columns[0] if state_base is None else f's_{state_base}'
    month_ref = months.columns[0] if month_base is None else f'm_{month_base}'
    const = pd.Series(1.0, index=frame.index, name='const')
    return pd.concat(
        [const, states.drop(columns=state_ref), months.drop(columns=month_ref)],
        axis=1,
    )


F_h2 = explicit_fe(h2)
d = h2.D.to_numpy(dtype=float)
assert np.linalg.matrix_rank(F_h2) == F_h2.shape[1] == 59

pure_fe = sm.OLS(h2.particularistic, F_h2).fit()

# D = F_h2 @ a, so setting beta to b subtracts b * a from the fixed-effects coefficients.
a = np.linalg.lstsq(F_h2, d, rcond=None)[0]
np.testing.assert_allclose(F_h2 @ a, d, atol=1e-12, rtol=0)

# Adding D also leaves the rank unchanged under the alternative reference categories.
for F in (F_h2, explicit_fe(h2, 'Puebla', 20140007)):
    rank_fe = np.linalg.matrix_rank(F)
    rank_full = np.linalg.matrix_rank(np.column_stack([F, d]))
    assert rank_fe == F.shape[1] == rank_full == 59

In [ ]:
constrained_models = {}
allocation = {'Fixed effects only': {'beta':np.nan, **pure_fe.params.to_dict()}}
for omitted, label in [(20160006,'Additionally omit June 2016'),(20140007,'Additionally omit July 2014')]:
    X = pd.concat([F_h2[['const']],h2.D.rename('D'),F_h2.drop(columns=['const',f'm_{omitted}'])],axis=1)
    assert np.linalg.matrix_rank(X) == X.shape[1] == 59
    fit = sm.OLS(h2.particularistic,X).fit(cov_type='cluster',
        cov_kwds={'groups':h2.state},use_t=True)
    constrained_models[label] = fit
    parameters = fit.params.to_dict()
    parameters[f'm_{omitted}'] = 0.
    parameters['beta'] = parameters.pop('D')
    allocation[label] = parameters
    assert np.max(np.abs(fit.fittedvalues-pure_fe.fittedvalues)) < 1e-12

coefficient_rows = [('β','beta'),('June 2016 γ','m_20160006'),
                    ('July 2014 γ','m_20140007'),('Puebla μ','s_Puebla'),('Intercept α','const')]
allocation_table = pd.DataFrame({name:{label:values.get(key,np.nan)*100
                      for label,key in coefficient_rows} for name,values in allocation.items()})
allocation_display = allocation_table.astype({'Fixed effects only': object})
allocation_display.at['β', 'Fixed effects only'] = 'Not included'
display(allocation_display.rename_axis('Parameter (pp)'))
constraint_inference = []
for label, fit in constrained_models.items():
    # Use the same clustered covariance and t reference distribution for the D contrast.
    treatment_contrast = fit.t_test('D = 0')
    treatment_se = treatment_contrast.sd.item()
    assert np.isfinite(treatment_se) and treatment_se > 0
    constraint_inference.append({'Additional constraint':label,'β (pp)':fit.params.D*100,
        'SE (pp)':treatment_se*100,'p (under constraint)':treatment_contrast.pvalue.item(),
        'Max. fitted-value difference from FE only':np.abs(fit.fittedvalues-pure_fe.fittedvalues).max()})
display(pd.DataFrame(constraint_inference))
assert np.isclose(constrained_models['Additionally omit June 2016'].params.D,.04599196022827533)
assert np.isclose(constrained_models['Additionally omit July 2014'].params.D,.05577290025185785)

In [ ]:
# Reconstruct the column order of the original command and retain linearly independent columns in sequence.
def redundant_fe_columns(matrix):
    basis, redundant = [], []
    for column in matrix.columns:
        residual = matrix[column].to_numpy(float).copy()
        for _ in range(2):  # Reorthogonalize to reduce rounding error.
            for q in basis:
                residual -= q * (q @ residual)
        norm = np.linalg.norm(residual)
        if norm < 1e-8:
            redundant.append(column)
        else:
            basis.append(residual / norm)
    return redundant


candidate = pd.concat([F_h2[['const']], h2.D.rename('D'),
                       F_h2.drop(columns='const')], axis=1)
omitted_columns = redundant_fe_columns(candidate)
assert omitted_columns == ['m_20160006']
print('Additional redundant column in the order-based reconstruction:', omitted_columns)

The author uses Stata's `reg` command for the final-year regression. When regressors are perfectly collinear, `reg` can omit a redundant column and continue, reporting the omission in the regression output. The public plotting command displays the treatment coefficient. The original run output would be needed to confirm which column Stata omitted in that run.

**Why $\beta$ equals the negative of the June 2016 fixed-effect coefficient**

Substituting the identity for $D$ into the original model gives

$$Y=\alpha+[\gamma_t+\beta(L_t+E_t)]+[\mu_s-\beta P_s]+\varepsilon.$$

The fixed-effects-only model is

$$Y=\alpha^{FE}+\gamma_t^{FE}+\mu_s^{FE}+\varepsilon^{FE}.$$

**Align the two parameterizations.** The state terms $\mu_s-\beta P_s$ and $\mu_s^{FE}$ both vary only by state. After aligning the state effects, let $c$ denote a constant shift. The time components satisfy

$$\alpha+\gamma_t+\beta(L_t+E_t)=\alpha^{FE}+\gamma_t^{FE}+c. \tag{1}$$

At the reference month $t_0$, both month effects are set to zero, $\gamma_{t_0}=\gamma^{FE}_{t_0}=0$. Evaluating (1) at $t_0$ yields

$$\alpha+\beta(L_{t_0}+E_{t_0})=\alpha^{FE}+c. \tag{2}$$

Subtract (2) from (1):

$$\gamma^{FE}_t=\gamma_t+\beta\big[(L_t+E_t)-(L_{t_0}+E_{t_0})\big]. \tag{3}$$

**The reference month determines the constant shift.** With $t_0$ in the early window, $L_{t_0}+E_{t_0}=1$, so (3) becomes

$$\gamma^{FE}_t=\gamma_t+\beta\big[(L_t+E_t)-1\big]. \tag{4}$$

For a month outside the two windows, such as June 2016, $L_t+E_t=0$. If its coefficient in the model retaining $D$ is constrained to zero, (4) gives

$$\gamma^{FE}_t=-\beta.$$

Using $\gamma^{FE}_{2016.6}=-0.04599196$ gives

$$\beta=+0.04599196\quad\text{(about +4.60 percentage points)}.$$

The reference month contributes a window indicator of 1, while June 2016 contributes 0. The difference of $-1$ is why the June coefficient equals $-\beta$ under the additional zero constraint.

### 4.6 Reconstruct Figure 6: Different final-year coefficients, identical fitted values

**Start with the fitted values for individual sessions.** In the final-year sample, $D=Fa$. For any specified $\beta$, change the fixed-effects-only coefficients $\widehat{\theta}$ to $\widehat{\theta}-\beta a$. For every session $i$, the fixed-effects adjustment exactly offsets the change in the treatment term:

$$\widehat Y_i(\beta)=F_i(\widehat{\theta}-\beta a)+\beta D_i=F_i\widehat{\theta}.$$

The left panel shows how the two components offset each other for one session with $D=1$. The right panel compares fitted values for all 1,511 sessions at four specified values of $\beta$: $-10$, $0$, $+4.60$, and $+10$ percentage points.

In [ ]:
june_beta = constrained_models['Additionally omit June 2016'].params.D
beta_pp = np.array([-10.0, 0.0, 100 * june_beta, 10.0])
fe_matrix = F_h2.to_numpy()
fe_estimate = pure_fe.params.to_numpy()
reference_fit = pure_fe.fittedvalues.to_numpy()
treatment = h2.D.to_numpy(dtype=float)

# Each column uses a specified beta. Each row keeps the same session.
beta_values = beta_pp / 100
adjusted_fe = fe_estimate[:, None] - a[:, None] * beta_values
all_fits = fe_matrix @ adjusted_fe + treatment[:, None] * beta_values
max_fit_gap = float(np.max(np.abs(all_fits - reference_fit[:, None])))
assert max_fit_gap < 1e-12

# Among sessions with D = 1, select the one whose baseline fit is closest to the group median.
treated_rows = np.flatnonzero(treatment == 1)
fit_median = np.median(reference_fit[treated_rows])
row = treated_rows[np.argmin(np.abs(reference_fit[treated_rows] - fit_median))]
beta_grid = np.linspace(beta_pp.min(), beta_pp.max(), 101)
treatment_part = treatment[row] * beta_grid
fe_part = 100 * (fe_matrix[row] @ (fe_estimate[:, None] - a[:, None] * beta_grid / 100))

fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4.7), layout='constrained')
left.plot(beta_grid, treatment_part, color='#2E5F7A', label='Treatment term')
left.plot(beta_grid, fe_part, color='#A34150', label='Intercept and fixed effects')
left.plot(beta_grid, treatment_part + fe_part, color='#222222', linewidth=2.1, label='Fitted value')
left.scatter(beta_pp, 100 * all_fits[row], color='#222222', s=25, zorder=3)
left.set_xticks(beta_pp, ['−10', '0', '+4.60', '+10'])
left.set(xlabel='Specified β (pp)', ylabel='Fitted particularistic share and components (pp)')
left.set_title('(a) Decomposition for one session (D = 1)', loc='left')
left.grid(axis='y', alpha=0.25)
left.legend(loc='upper center', bbox_to_anchor=(0.5, -0.17), ncol=3, frameon=False)

limits = [100 * min(reference_fit.min(), all_fits.min()),
          100 * max(reference_fit.max(), all_fits.max())]
margin = 0.04 * (limits[1] - limits[0])
limits = [limits[0] - margin, limits[1] + margin]
right.plot(limits, limits, color='#777777', linestyle='--', linewidth=1, zorder=1)
right.scatter(100 * np.repeat(reference_fit, len(beta_pp)), 100 * all_fits.ravel(),
              color='#2E5F7A', alpha=0.3, s=9, linewidths=0, zorder=2)
right.set(xlim=limits, ylim=limits, xlabel='Fitted particularistic share, FE-only model (%)',
          ylabel='Fitted particularistic share at specified β (%)')
right.set_aspect('equal', adjustable='box')
right.set_title(f'(b) Comparison across {len(h2):,} sessions', loc='left')
right.text(0.04, 0.96, f'All four sets lie on y = x\nMaximum absolute difference: {max_fit_gap:.1e}',
           transform=right.transAxes, va='top', fontsize=11)
plt.show()

**Reconstruct the public script.** The author fits the model separately for `timinga`, `timingb`, and `timingc` and plots 90% and 95% clustered confidence intervals. The final-year command is

```stata
reg particularistic timb3 i.edo i.date2 if timingc==1, robust cluster(edo)
```

The next cell translates all three regressions into Python. For the final year, it uses the column-order reconstruction in Section 4.5 and additionally omits June 2016, which reproduces the plotted +4.60 percentage points.

**Reconstruct the estimate from a linear combination of fixed effects.** Let $F$ be the fixed-effects matrix including the intercept, let $D=Fa$, and let $\widehat{\theta}$ be the fixed-effects-only coefficient vector. For any $\beta$, the fitted values remain $F\widehat{\theta}$ when the fixed-effects coefficients are set to $\widehat{\theta}-\beta a$. Let $c$ select the difference between the coefficient for month $m$ and the original reference month $t_0$. Setting that difference to zero gives

$$c^\top(\widehat{\theta}-\beta a)=0
\quad\Longrightarrow\quad
\widehat{\beta}=\frac{c^\top\widehat{\theta}}{c^\top a}.$$

For June 2016 relative to July 2012, the denominator is $-1$, giving $\widehat{\beta}=-\widehat{\gamma}^{FE}_{2016.6}$. With a different reference month in the fixed-effects-only model, the same quantity is $\widehat{\gamma}^{FE}_{2012.7}-\widehat{\gamma}^{FE}_{2016.6}$. Both contrasts reproduce the final-year point and confidence intervals in Figure 6. July 2014 and January 2015 provide two alternative constraints for comparison.

In [ ]:
def fit_figure6_ols(frame, matrix):
    return sm.OLS(frame.particularistic, matrix).fit(
        cov_type='cluster', cov_kwds={'groups': frame.state}, use_t=True)


def figure6_inference(fit, contrast):
    test = fit.t_test(contrast)
    return {'beta': test.effect.item(), 'se': test.sd.item(),
            'df': fit.df_resid_inference}


# Translate the three original regressions. Apply the column selection from Section 4.5 in the final year.
figure6_stata = {}
for period in ('timinga', 'timingb', 'timingc'):
    frame = h2_designs[period].frame
    fe = explicit_fe(frame)
    design = pd.concat([fe[['const']], frame.D_long.rename('D'),
                        fe.drop(columns='const')], axis=1)
    redundant = redundant_fe_columns(design)
    if period == 'timingc':
        assert redundant == omitted_columns
    assert 'D' not in redundant
    design = design.drop(columns=redundant)
    assert np.linalg.matrix_rank(design) == design.shape[1]
    figure6_stata[period] = fit_figure6_ols(frame, design)

figure6_reconstructed = [figure6_inference(fit, 'D = 0')
                         for fit in figure6_stata.values()]
for result, check in zip(figure6_reconstructed[:2], h2_records[:2]):
    np.testing.assert_allclose([result['beta'], result['se']],
                               [check['b'], check['se']], atol=1e-12, rtol=0)


def figure6_from_fe(fe, omitted_month):
    fit = fit_figure6_ols(h2, fe)
    d_as_fe = np.linalg.lstsq(fe, h2.D, rcond=None)[0]
    np.testing.assert_allclose(fe @ d_as_fe, h2.D, atol=1e-12, rtol=0)

    # c contrasts the constrained month with the original reference month. An omitted reference coefficient is zero.
    c = np.array([int(col == f'm_{omitted_month}') - int(col == 'm_20120007')
                  for col in fe.columns], dtype=float)
    denominator = c @ d_as_fe
    assert np.isclose(denominator, -1.0)
    result = figure6_inference(fit, c / denominator)
    shifted_fe = fit.params.to_numpy() - result['beta'] * d_as_fe
    prediction = fe @ shifted_fe + result['beta'] * h2.D
    assert np.isclose(c @ shifted_fe, 0.0, atol=1e-12)
    return result, prediction


june_drop = constrained_models['Additionally omit June 2016']
july_drop = constrained_models['Additionally omit July 2014']
june_fe, june_prediction = figure6_from_fe(F_h2, 20160006)
june_rebased, rebased_prediction = figure6_from_fe(
    explicit_fe(h2, 'Puebla', 20140007), 20160006)
july_fe, july_prediction = figure6_from_fe(F_h2, 20140007)
january_fe, january_prediction = figure6_from_fe(F_h2, 20150001)

figure6_comparisons = [
    ('Original script reconstruction', figure6_reconstructed[-1], figure6_stata['timingc'].fittedvalues),
    ('Retain D, omit June 2016', figure6_inference(june_drop, 'D = 0'), june_drop.fittedvalues),
    ('FE only: June 2016 contrast', june_fe, june_prediction),
    ('New reference: same June contrast', june_rebased, rebased_prediction),
    ('Constrain July 2014', july_fe, july_prediction),
    ('Constrain January 2015', january_fe, january_prediction),
]
for _, result, prediction in figure6_comparisons:
    np.testing.assert_allclose(prediction, pure_fe.fittedvalues, atol=1e-12, rtol=0)
    assert result['df'] == 13
reference = [figure6_reconstructed[-1]['beta'], figure6_reconstructed[-1]['se']]
for _, result, _ in figure6_comparisons[:4]:
    np.testing.assert_allclose([result['beta'], result['se']], reference, atol=1e-12, rtol=0)
july_check = figure6_inference(july_drop, 'D = 0')
np.testing.assert_allclose([july_fe['beta'], july_fe['se']],
                           [july_check['beta'], july_check['se']], atol=1e-12, rtol=0)

january_design = F_h2.drop(columns='m_20150001').assign(D=h2.D)
assert np.linalg.matrix_rank(january_design) == january_design.shape[1] == 59
january_drop = fit_figure6_ols(h2, january_design)
january_check = figure6_inference(january_drop, 'D = 0')
np.testing.assert_allclose([january_fe['beta'], january_fe['se']],
                           [january_check['beta'], january_check['se']], atol=1e-12, rtol=0)
np.testing.assert_allclose(january_drop.fittedvalues, pure_fe.fittedvalues, atol=1e-12, rtol=0)

display(pd.DataFrame([
    {'Panel': f'({letter})', 'Final-year parameterization': label, 'Coefficient (pp)': 100 * result['beta'],
     'Clustered SE (pp)': 100 * result['se'],
     'Max. fitted-value difference from FE only': np.max(np.abs(prediction - pure_fe.fittedvalues))}
    for letter, (label, result, prediction) in zip('abcdef', figure6_comparisons, strict=True)
]))

In [ ]:
from matplotlib.lines import Line2D

fig, axes = plt.subplots(2, 3, figsize=(13, 7.8), sharey=True, layout='constrained')
max_comparison_gap = max(float(np.max(np.abs(prediction - pure_fe.fittedvalues)))
                         for _, _, prediction in figure6_comparisons)
assert max_comparison_gap < 1e-12
fig.suptitle(f'Identical fitted values for all 1,511 sessions',
             fontsize=14)
for i, (ax, (label, last, _)) in enumerate(zip(axes.flat, figure6_comparisons, strict=True)):
    title = f'({chr(97 + i)}) {label}'
    last_color = '#2E5F7A' if i < 4 else '#A34150'
    ax.axhline(0, color='#777777', linestyle='--', linewidth=1)
    for position, result in enumerate([*figure6_reconstructed[:2], last]):
        estimate = 100 * result['beta']
        se = 100 * result['se']
        color = last_color if position == 2 else '#686868'
        for confidence, width in [(0.95, 1.2), (0.90, 3.0)]:
            half_width = student_t.ppf((1 + confidence) / 2, result['df']) * se
            ax.vlines(position, estimate - half_width, estimate + half_width,
                      color=color, linewidth=width)
        ax.scatter(position, estimate, color=color, marker='s', zorder=3, s=34)
    ax.annotate(f'{100 * last["beta"]:+.2f}', (2, 100 * last['beta']),
                xytext=(9, 0), textcoords='offset points', va='center', color=last_color)
    ax.set_xticks([0, 1, 2], ['Election (t)', 't+1', 't+2'])
    ax.set_xlim(-0.35, 2.55)
    ax.set_title(title, loc='left', fontsize=13)
    ax.grid(axis='y', alpha=0.22)
for ax in axes[:, 0]:
    ax.set_ylabel('Coefficient (pp)')
fig.legend(handles=[Line2D([0], [0], color='#686868', linewidth=1.2, label='95% clustered confidence interval'),
                    Line2D([0], [0], color='#686868', linewidth=3.0, label='90% clustered confidence interval')],
           loc='outside lower center', ncol=2, frameon=False)
plt.show()

Panels (a)–(d) reproduce the same final-year estimate of **+4.60 percentage points**. Panel (d) changes the reference month of the fixed-effects-only regression to July 2014 and evaluates the same contrast. Panels (e) and (f) use July 2014 and January 2015 as the additional constraint, respectively, which changes the final-year coefficient and interval. The fitted values for all 1,511 final-year sessions agree across the six panels to floating-point precision. Panels (a)–(f) correspond to the rows of the table.

### 4.7 Restore missing sample coverage

The perfect collinearity comes from gaps in the observed comparisons. Mexico City and Puebla, the only two areas in the term-limited group, each lack observations in key windows. Restoring a missing state-month combination can break the collinearity.

In [ ]:
def rank_check(frame):
    X = explicit_fe(frame)
    return int(np.linalg.matrix_rank(X)), int(np.linalg.matrix_rank(np.column_stack([X,frame.D])))

minimal = h2[['state','date2','D']].copy()
hypothetical = pd.concat([minimal,pd.DataFrame([{'state':'CdMex','date2':20170007,'D':0.}])],ignore_index=True)
puebla_actual = data.loc[data.state.eq('Puebla') & data.date2.eq(20140007)].copy()
puebla_actual['D'] = puebla_actual.D_long
assert len(puebla_actual) == 5 and puebla_actual.D.eq(0).all()
restored = pd.concat([minimal,puebla_actual[['state','date2','D']]],ignore_index=True)
checks = [
    {'Sample':'Original H2 final-year sample','N':len(minimal),'Added observations':'None','Rank of FE':rank_check(minimal)[0],'Rank of FE + D':rank_check(minimal)[1]},
    {'Sample':'Add Mexico City, July 2017','N':len(hypothetical),'Added observations':'1 hypothetical observation',
     'Rank of FE':rank_check(hypothetical)[0],'Rank of FE + D':rank_check(hypothetical)[1]},
    {'Sample':'Restore Puebla, July 2014','N':len(restored),'Added observations':'5 observed records',
     'Rank of FE':rank_check(restored)[0],'Rank of FE + D':rank_check(restored)[1]}
]
assert rank_check(hypothetical)==(59,60) and rank_check(restored)==(59,60)
display(pd.DataFrame(checks))
display(pd.DataFrame([
 {'Added state-month':'Mexico City, July 2017 (hypothetical)','R×T':0,'L+E-P':1},
 {'Added state-month':'Puebla, July 2014 (observed)','R×T':0,'L+E-P':-1}]))

**H2 erratum.** For the 1,511 sessions selected by the original script, the fixed effects exactly represent $D$. Adding $D$ after the usual reference categories are omitted does not increase the matrix rank. Changing $\beta$ and adjusting the fixed effects accordingly leaves the fitted values unchanged. The final-year effect shown in Figure 6 is therefore not separately identified under the full specification.

<a id="section-5"></a>

## 5. Supplementary analysis: Short-term reelection vs. term limits

Table 2 compares the long-term reelection and term-limited groups. Table 3 compares the long-term and short-term reelection groups. A direct short-term versus term-limited comparison asks how the outcome changed when only one additional term was available. The point estimate in this supplementary comparison is negative, which qualifies the broader hypothesis about reelection incentives.

The treatment variable is `treat2 × reform`. The regression follows the original coding and uses state and month-by-year fixed effects. The sample includes six states in the short-term reelection group and two areas in the term-limited group, for a total of eight clusters. Both the small number of clusters and the limited set of comparison areas matter for inference.

### 5.1 Point estimates, inference with few clusters, and excluding control areas

In [ ]:
short_results = []
for label, excluded in [('Full sample',None),('Exclude Mexico City','CdMex'),('Exclude Puebla','Puebla')]:
    frame = data.loc[data.D_short.notna()].copy()
    if excluded:
        frame = frame.loc[frame.state.ne(excluded)]
    model = FixedEffectsDesign(frame)
    r = model.fit('D_short', wild=True)
    r['comparison'] = label
    short_results.append(r)
assert short_results[0]['n'] == 2807
assert np.isclose(short_results[0]['b'], -.019597792144748865)
assert short_results[0]['wild_p'] == .0625
display(result_table(short_results))
fig, ax = plt.subplots(figsize=(10,4),layout='constrained')
forest(ax, short_results, ['Full sample','Exclude Mexico City','Exclude Puebla'],
       'Supplement: Short-term reelection vs. term limits', [COLORS['short']]*3)
plt.show()

The full-sample point estimate is about **−1.96 percentage points**. The conventional state-clustered p-value is about 0.0019, and the enumerated wild cluster bootstrap p-value is **0.0625**. The bootstrap evidence is weaker after excluding either comparison area.

### 5.2 Pre-treatment cutoffs and multiple comparisons

The analysis uses the same 33 cutoffs and only observations from before the reform. If every test has a 5% false positive rate and all null hypotheses are true, the expected number of significant results is $33\times0.05=1.65$.

The joint bootstrap uses the same state ordering and weights across cutoffs and tests the maximum absolute t-statistic.

In [ ]:
short_placebos, short_joint_p = placebo_checks('treat2', wild=True)
significant = short_placebos.p.lt(.05)
assert len(short_placebos) == 33 and significant.sum() == 6
assert (short_placebos.loc[significant,'b'] < 0).all()
assert np.isclose(short_joint_p, .1171875)
display(pd.DataFrame({'Cutoffs':[33], 'Conventionally significant cutoffs':[int(significant.sum())],
                       'Joint max-|t| bootstrap p':[short_joint_p]}))
display(short_placebos[['cutoff','cohort_shift','month','b','se','p','wild_p']].rename(
    columns={'cutoff':'Cutoff index','cohort_shift':'Years before treatment','month':'Month',
             'b':'Coefficient (proportion units)','se':'Clustered SE','p':'Conventional p','wild_p':'bootstrap p'}))
fig, ax = plt.subplots(figsize=(11,4),layout='constrained')
for _, row in short_placebos.iterrows():
    color = COLORS['short'] if row.p < .05 else COLORS['muted']
    ax.errorbar(row.cutoff,row.b*100,
        yerr=[[100*(row.b-row.ci_lower)],[100*(row.ci_upper-row.b)]],
        fmt='o' if row.p<.05 else 's',ms=4,capsize=2,color=color)
ax.axhline(0,color='black',ls='--',lw=1)
ax.set(xlabel='Placebo cutoff (original order)',ylabel='DID estimate (pp)',
       xticks=[1,6,12,18,24,30,33])
ax.set_title(f'Short-term reelection vs. term limits: Pre-treatment checks, joint p = {short_joint_p:.3f}',loc='left')
plt.show()

DID relies on the parallel-trends assumption. Negative pre-treatment placebo estimates could attenuate a positive main estimate if the corresponding trend persisted into the treatment period. In the short-term versus term-limited comparison, some negative placebo estimates have the same sign as the negative main estimate. This makes the negative estimate harder to interpret as a treatment effect. The parallel-trends assumption itself applies regardless of the sign of the estimate.

The article does not report a direct short-term versus term-limited DID estimate, and the public analysis and appendix scripts do not contain a corresponding standalone outcome regression. The scripts use `treat2` in constructing the H3 comparison, whose treatment term is `c.treat42#c.reform`. The public materials leave open why the direct comparison was not reported.